# 🌐 REST API Data Collection — Cryptocurrency & Weather Data

> **A beginner-friendly, step-by-step guide to calling public REST APIs in Python.**  
> We fetch live crypto market data from **CoinGecko** and historical weather data from **Open-Meteo**,  
> then save everything to CSV for downstream analysis.

---

**Author:** Raj Shewale · [Kaggle](https://www.kaggle.com/shambhushewale) · [GitHub](https://github.com/Precipitation-Rain)  
**APIs used:** [CoinGecko](https://www.coingecko.com/en/api) (free, no key needed) · [Open-Meteo](https://open-meteo.com/) (free, no key needed)  
**Libraries:** `requests`, `pandas`, `time`


## 📋 Table of Contents

1. [What is a REST API?](#section-1)
2. [Quick Exploration — First API Call](#section-2)
3. [Clean Production API Call](#section-3)
4. [Paginated API — Fetching All Pages](#section-4)
5. [Weather Data — Open-Meteo API](#section-5)
   - 5a. [Satara — Daily Weather Data (2025)](#section-5a)
   - 5b. [Ishwarpur — Hourly Weather Data (2020–2025)](#section-5b)


<a id='section-1'></a>
## 🔌 Section 1 — What is a REST API?

A **REST API** (Representational State Transfer) is a standard way for applications to communicate over the internet.  
Think of it like ordering at a restaurant:

| Restaurant | API |
|------------|-----|
| You (customer) | Your Python code |
| Waiter | HTTP request (`requests.get`) |
| Kitchen | Server / API |
| Food delivered | JSON response |

**Key concepts:**

- **Endpoint** — the URL you call (like the menu category)
- **Query Parameters** — filters you attach to the URL (`?vs_currency=usd&per_page=100`)
- **Status Code** — `200` = OK, `429` = rate limited, `400` = bad request
- **JSON** — the format the server sends data back in (Python reads it as a dict/list)


<a id='section-2'></a>
## 🔍 Section 2 — Quick Exploration: First API Call

Before writing clean code, let's explore the API interactively — one step at a time.  
We'll use the **CoinGecko `/coins/markets`** endpoint to get live prices for the top 100 cryptocurrencies.

> **No API key required.** CoinGecko's public tier is completely free.


In [ ]:
# Standard imports for API work
import requests
import pandas as pd
import time

In [ ]:
# ── ENDPOINT ──────────────────────────────────────────────────────────────
# This is the URL where CoinGecko serves market data for all coins
url = "https://api.coingecko.com/api/v3/coins/markets"

# ── QUERY PARAMETERS ──────────────────────────────────────────────────────
# Instead of hard-coding ?vs_currency=usd&order=... in the URL,
# we pass a dictionary — requests adds it automatically
parameters = {
    "vs_currency": "usd",         # Return prices in USD
    "order":       "market_cap_desc", # Sort by largest market cap first
    "per_page":    100,            # Return 100 coins per page
    "page":        1,             # Start from page 1
    "sparkline":   False          # Skip sparkline chart data
}

response = requests.get(url=url, params=parameters)
print(f"Status code : {response.status_code}")   # 200 = success
print(f"Full URL    : {response.url}")            # See what requests built

Let's inspect the raw response — what type it is, how many coins we got, and what a single coin record looks like.


In [ ]:
data = response.json()        # Parse JSON → Python list

print(f"Type : {type(data)}")   # Should be <class 'list'>
print(f"Count: {len(data)}")    # Should be 100


In [ ]:
# Peek at the first coin's raw data
data[0]

In [ ]:
# Convert the list of dicts → tidy DataFrame
df_explore = pd.DataFrame(data)
print(f"Shape: {df_explore.shape}")
df_explore.head()

<a id='section-3'></a>
## ✅ Section 3 — Clean Production API Call

Now that we understand the structure, here's the same call written cleanly —  
with clear comments, proper status checking, and CSV export.

This is the pattern you'd use in a real data pipeline.


In [ ]:
import requests
import pandas as pd

# ── ENDPOINT ──────────────────────────────────────────────────────────────
url = "https://api.coingecko.com/api/v3/coins/markets"

# ── QUERY PARAMETERS ──────────────────────────────────────────────────────
# Passing as a dict lets requests auto-encode them into the URL
params = {
    "vs_currency": "usd",
    "order":       "market_cap_desc",
    "per_page":    100,
    "page":        1,
    "sparkline":   False
}

# ── REQUEST ───────────────────────────────────────────────────────────────
response = requests.get(url, params=params)
print(f"Status code : {response.status_code}")
print(f"Full URL    : {response.url}")

# ── STATUS CHECK ──────────────────────────────────────────────────────────
if response.status_code != 200:
    raise Exception(f"API call failed: {response.status_code} — {response.text}")

# ── PARSE → DATAFRAME → CSV ───────────────────────────────────────────────
data = response.json()
print(f"Coins received : {len(data)}")     # Should be 100

df_single = pd.DataFrame(data)
print(f"Shape          : {df_single.shape}")
print(f"Columns        : {df_single.columns.tolist()}")

df_single.to_csv("crypto_market_top100.csv", index=False)
print("\n✅ Saved → crypto_market_top100.csv")

df_single.head()

<a id='section-4'></a>
## 🔄 Section 4 — Paginated API: Fetching All Pages

CoinGecko lists thousands of coins, but serves them **100-250 per page**.  
To collect everything, we loop through all pages until the API returns an empty list.

**Rate limiting:** CoinGecko's free tier allows ~10-30 requests/minute.  
We handle the `429 Too Many Requests` response by waiting 30 seconds and retrying,  
and add a 2-second sleep between normal requests to stay within limits.


In [ ]:
import requests
import pandas as pd
import time

url      = "https://api.coingecko.com/api/v3/coins/markets"
page     = 1
all_data = []

print("Starting paginated fetch...\n")

while True:
    params = {
        "vs_currency": "usd",
        "order":       "market_cap_desc",
        "per_page":    250,   # max allowed per page
        "page":        page,
        "sparkline":   False
    }

    response = requests.get(url, params=params)
    status   = response.status_code

    # ── Rate limit hit — wait and retry ────────────────────────────────
    if status == 429:
        print("⚠️  Rate limited (429) — waiting 30 seconds...")
        time.sleep(30)
        continue            # retry the same page

    # ── Any other error — stop ──────────────────────────────────────────
    if status != 200:
        print(f"❌ Unexpected error {status} — stopping.")
        break

    data = response.json()

    # ── Empty page means we've collected everything ─────────────────────
    if len(data) == 0:
        print("\n✅ All pages collected — no more data.")
        break

    all_data.extend(data)
    print(f"Page {page:>3} → {len(data):>3} coins received  |  Total so far: {len(all_data):,}")

    page += 1
    time.sleep(2)           # polite delay between requests


In [ ]:
# ── Build DataFrame & Export ──────────────────────────────────────────────
df_all = pd.DataFrame(all_data)

print(f"Total coins collected : {len(df_all):,}")
print(f"Columns               : {len(df_all.columns)}")

df_all.to_csv("crypto_market_all_coins.csv", index=False)
print("\n✅ Saved → crypto_market_all_coins.csv")

df_all.head()

<a id='section-5'></a>
## 🌤️ Section 5 — Weather Data: Open-Meteo API

[Open-Meteo](https://open-meteo.com/) provides **free historical and forecast weather data** — no API key required.  
It offers multiple endpoints: historical archive, forecast, air quality, marine, and climate projections.

**Key parameters:**

| Parameter | Description |
|-----------|-------------|
| `latitude` / `longitude` | Location coordinates |
| `start_date` / `end_date` | Date range (YYYY-MM-DD) |
| `daily` / `hourly` | List of variables to fetch |
| `timezone` | E.g. `Asia/Kolkata` — dates returned in local time |

We'll collect data for two locations in Maharashtra:
- **Satara** — Daily summary data for 2025
- **Uran-Ishwarpur** — Hourly granular data for 2020–2025


<a id='section-5a'></a>
### 📍 Section 5a — Satara: Daily Weather Data (2025)

Daily data gives one row per day — max/min temperature, precipitation totals, wind, radiation, etc.  
This is ideal for trend analysis and monthly/seasonal comparisons.

**Location:** Satara, Maharashtra · Lat: 17.6859 · Lon: 73.9933


In [ ]:
import requests
import pandas as pd

# ── ENDPOINT (historical archive) ─────────────────────────────────────────
url = "https://archive-api.open-meteo.com/v1/archive"

# ── PARAMETERS ────────────────────────────────────────────────────────────
params = {
    "latitude":   17.68589,
    "longitude":  73.99333,
    "start_date": "2025-01-01",
    "end_date":   "2025-12-31",
    "daily": [
        "weather_code",
        "temperature_2m_max",
        "temperature_2m_min",
        "apparent_temperature_max",
        "apparent_temperature_min",
        "sunrise",
        "sunset",
        "daylight_duration",
        "sunshine_duration",
        "precipitation_sum",
        "rain_sum",
        "snowfall_sum",
        "precipitation_hours",
        "wind_speed_10m_max",
        "wind_gusts_10m_max",
        "wind_direction_10m_dominant",
        "shortwave_radiation_sum",
        "et0_fao_evapotranspiration"
    ],
    "timezone": "Asia/Kolkata"
}

# ── REQUEST ───────────────────────────────────────────────────────────────
response = requests.get(url, params=params)
print(f"Status: {response.status_code}")

if response.status_code != 200:
    raise Exception(f"API Error: {response.status_code} — {response.text}")

# ── PARSE → DATAFRAME ─────────────────────────────────────────────────────
data          = response.json()
df_satara     = pd.DataFrame(data["daily"])

print(f"Records  : {len(df_satara)} days")
print(f"Columns  : {len(df_satara.columns)}")

# ── EXPORT ────────────────────────────────────────────────────────────────
df_satara.to_csv("Satara_Daily_Weather_2025.csv", index=False)
print("\n✅ Saved → Satara_Daily_Weather_2025.csv")

df_satara.head(10)

In [ ]:
# Quick stats on the Satara dataset
df_satara.describe().round(2)

<a id='section-5b'></a>
### 📍 Section 5b — Uran-Ishwarpur: Hourly Weather Data (2020–2025)

Hourly data gives one row per hour — perfect for granular analysis like  
peak temperature times, humidity patterns, and short-term event detection.

**Location:** Uran-Ishwarpur, Maharashtra · Lat: 17.6805 · Lon: 74.0183  
**Range:** January 2020 → December 2025 (~52,584 hourly records)

> ⏱️ **Note:** This call fetches 6 years of hourly data across 31 variables —  
> expect a slightly longer response time (~10–30 seconds).


In [ ]:
import requests
import pandas as pd

# ── ENDPOINT ──────────────────────────────────────────────────────────────
url = "https://archive-api.open-meteo.com/v1/archive"

# ── PARAMETERS ────────────────────────────────────────────────────────────
params = {
    "latitude":   17.6805,
    "longitude":  74.0183,
    "start_date": "2020-01-01",
    "end_date":   "2025-12-31",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "dew_point_2m",
        "apparent_temperature",
        "precipitation",
        "rain",
        "snowfall",
        "snow_depth",
        "weather_code",
        "pressure_msl",
        "surface_pressure",
        "cloud_cover",
        "cloud_cover_low",
        "cloud_cover_mid",
        "cloud_cover_high",
        "wind_speed_10m",
        "wind_speed_100m",
        "wind_direction_10m",
        "wind_direction_100m",
        "wind_gusts_10m",
        "soil_temperature_0_to_7cm",
        "soil_temperature_7_to_28cm",
        "soil_temperature_28_to_100cm",
        "soil_temperature_100_to_255cm",
        "soil_moisture_0_to_7cm",
        "soil_moisture_7_to_28cm",
        "soil_moisture_28_to_100cm",
        "soil_moisture_100_to_255cm",
        "shortwave_radiation",
        "vapour_pressure_deficit",
        "et0_fao_evapotranspiration"
    ],
    "timezone": "Asia/Kolkata"
}

# ── REQUEST ───────────────────────────────────────────────────────────────
print("Fetching 6 years of hourly data...")
response = requests.get(url, params=params)
print(f"Status: {response.status_code}")

if response.status_code != 200:
    raise Exception(f"API Error: {response.status_code} — {response.text}")

# ── PARSE → DATAFRAME ─────────────────────────────────────────────────────
data             = response.json()
df_ishwarpur     = pd.DataFrame(data["hourly"])

print(f"Records  : {len(df_ishwarpur):,} hourly rows")
print(f"Columns  : {len(df_ishwarpur.columns)}")

# ── EXPORT ────────────────────────────────────────────────────────────────
df_ishwarpur.to_csv("Ishwarpur_Hourly_Weather_2020_2025.csv", index=False)
print("\n✅ Saved → Ishwarpur_Hourly_Weather_2020_2025.csv")

df_ishwarpur.head(10)

In [ ]:
# Quick stats on the Ishwarpur dataset
df_ishwarpur.describe().round(2)

---
## 🎯 Summary

| Dataset | Source | Frequency | Records | File |
|---------|--------|-----------|---------|------|
| Crypto Top 100 | CoinGecko | Snapshot | 100 coins | `crypto_market_top100.csv` |
| Crypto All Coins | CoinGecko | Snapshot (paginated) | ~12,000+ coins | `crypto_market_all_coins.csv` |
| Satara Weather | Open-Meteo | Daily (2025) | 365 days | `Satara_Daily_Weather_2025.csv` |
| Ishwarpur Weather | Open-Meteo | Hourly (2020–2025) | ~52,584 rows | `Ishwarpur_Hourly_Weather_2020_2025.csv` |

**Key patterns covered:**
- Single API call with query parameters
- Response status checking
- JSON → DataFrame → CSV pipeline
- Pagination loop with rate-limit handling (`429` retry)
- Multi-variable weather API with daily and hourly granularity

---
*Data collected using public APIs. CoinGecko and Open-Meteo are free for non-commercial use.*
